# 05 - Retrieval-depth evaluation v3

## Purpose

This notebook records the adaptive version 3 retrieval-depth evaluation. The
version 2 adequacy gate showed that the candidate range through rank 20 was
still insufficient, because direct evidence first appeared in the rank 21-30
diagnostic tail for two questions.

Version 3 is the final planned candidate-range expansion. It has three
deliberately separate actions:

1. reuse the exact version 2 query matrix, search locally to rank 40 and
   migrate the 600 verified rank 1-30 judgements;
2. judge only the 200 new rank 31-40 candidate pairs through the same blinded
   human interface;
3. score all 800 judgements offline using the unchanged selection rule and the
   terminal version 3 adequacy gate.

This separation keeps the adaptive decision visible, preserves the completed
human evidence and prevents the new diagnostic results from influencing the
protocol that produced them.


## Evaluation and leakage boundary

The following boundaries apply throughout this notebook:

- Version 3 follows the protocol amendment frozen in commit `5c8e0b9`.
- The corrected version 3 configuration contracts were frozen before the
  expanded search, with shared-schema compatibility recorded in commit
  `0a515f7`.
- The 20 questions, query embeddings, corpus chunks and FAISS index are
  unchanged.
- The query matrix is reused byte for byte, so version 3 makes no embedding
  API request.
- The version 3 pool contains ranks 1-40. Candidate depths end at 30 and ranks
  31-40 form the final diagnostic tail.
- The 600 rank 1-30 grades and notes are migrated only after exact candidate
  equality checks pass.
- Only the 200 new rank 31-40 records are presented for additional human
  judgement.
- Retrieval rank, similarity score, candidate-depth membership, identifiers
  and provenance remain hidden from the annotation display.
- No model-generated answer or AI-generated relevance recommendation is used
  during annotation.
- Ground-truth answers remain unavailable during retrieval, annotation and
  scoring.
- The same 20 benchmark questions are used for calibration and later model
  evaluation, so this limitation must remain explicit in the dissertation.


In [1]:
from __future__ import annotations

from html import escape
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import os
import sys

from IPython.display import HTML, clear_output, display
import numpy as np


def locate_project_root() -> Path:
    """Locate the repository from its root or notebooks directory."""

    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for candidate in candidates:
        if (
            (candidate / "pyproject.toml").is_file()
            and (candidate / "configs/retrieval-evaluation-config-v3.json").is_file()
        ):
            return candidate
    raise RuntimeError(
        "Run this notebook from the repository root or its notebooks directory."
    )


def resolve_project_path(root: Path, relative_path: str) -> Path:
    """Resolve a configured path without allowing it to escape the project."""

    path = (root / relative_path).resolve()
    if not path.is_relative_to(root):
        raise RuntimeError(f"Configured path leaves the project: {relative_path}")
    return path


def env_file_defines_key(path: Path, key: str) -> bool:
    """Check whether a key name exists without reading or printing its value."""

    if not path.is_file():
        return False
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        name = line.split("=", 1)[0].strip()
        if name.startswith("export "):
            name = name.removeprefix("export ").strip()
        if name == key:
            return True
    return False


PROJECT_ROOT = locate_project_root()
SOURCE_ROOT = (PROJECT_ROOT / "src").resolve()
if not SOURCE_ROOT.is_dir():
    raise RuntimeError(f"Project source directory is missing: {SOURCE_ROOT}")

source_root_string = str(SOURCE_ROOT)
if source_root_string not in sys.path:
    sys.path.insert(0, source_root_string)

from geotech_rag.retrieval_evaluation import (
    calculate_file_sha256,
    load_retrieval_evaluation_config,
    load_retrieval_evaluation_inputs,
    score_retrieval_judgements,
    serialise_json_lines,
)
from geotech_rag.retrieval_evaluation_v3 import (
    prepare_retrieval_evaluation_v3,
)


CONFIG_PATH = PROJECT_ROOT / "configs/retrieval-evaluation-config-v3.json"
LINEAGE_PATH = PROJECT_ROOT / "configs/retrieval-evaluation-v3-lineage.json"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks/05_retrieval_depth_evaluation_v3.ipynb"

print("Project root:", PROJECT_ROOT)
print("Source root added to Python path:", SOURCE_ROOT)
print("Configuration:", CONFIG_PATH.relative_to(PROJECT_ROOT))
print("Lineage:", LINEAGE_PATH.relative_to(PROJECT_ROOT))
print("Ground truth accessed: False")


Project root: /home/zaki/coding_F/GismaProjects/Thesis
Source root added to Python path: /home/zaki/coding_F/GismaProjects/Thesis/src
Configuration: configs/retrieval-evaluation-config-v3.json
Lineage: configs/retrieval-evaluation-v3-lineage.json
Ground truth accessed: False


## 1. Frozen v3 protocol and input preflight

This preflight loads the committed version 3 configuration and lineage
contract. It validates the frozen input fingerprints, the expanded candidate
range, the diagnostic-tail boundary and the planned judgement reuse counts.

The cell performs no API request and does not display question text, chunk
text, grades or evidence notes.


In [2]:
# Load the isolated v3 configuration and its frozen v2-to-v3 lineage.
config = load_retrieval_evaluation_config(CONFIG_PATH)
inputs = load_retrieval_evaluation_inputs(CONFIG_PATH, PROJECT_ROOT)
lineage = json.loads(LINEAGE_PATH.read_text(encoding="utf-8"))
outputs = config["outputs"]

# Resolve every output through the same project-boundary check used in v1.
QUERY_MATRIX_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["query_embedding_matrix_relative_path"],
)
CANDIDATE_POOL_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["ranked_candidate_pool_relative_path"],
)
JUDGEMENT_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["blinded_judgement_file_relative_path"],
)
SUMMARY_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["summary_relative_path"],
)
METRICS_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["metrics_relative_path"],
)
METRICS_TABLE_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["metrics_table_relative_path"],
)

# Derive the expected counts from the frozen contracts rather than inserting
# independent values into the notebook.
question_count = len(inputs.question_records)
pool_depth = config["search"]["judgement_pool_depth"]
expected_candidate_count = question_count * pool_depth
expected_reused_count = lineage["judgement_reuse"][
    "reused_judgement_count"
]
expected_new_count = lineage["judgement_reuse"][
    "new_judgement_count"
]

if (
    lineage["target_v3"]["configuration_id"]
    != config["configuration_id"]
):
    raise RuntimeError(
        "The v3 configuration and lineage identifiers differ."
    )

if (
    lineage["target_v3"]["configuration_sha256"]
    != calculate_file_sha256(CONFIG_PATH)
):
    raise RuntimeError(
        "The v3 configuration fingerprint differs from the lineage."
    )

if expected_reused_count + expected_new_count != expected_candidate_count:
    raise RuntimeError(
        "The v3 reuse counts do not equal the candidate count."
    )

print("Retrieval-evaluation v3 preflight:")
print("  Configuration ID:", config["configuration_id"])
print("  Questions:", question_count)
print("  Corpus chunks:", len(inputs.chunk_records))
print("  FAISS vectors:", inputs.faiss_index.ntotal)
print("  Candidate depths:", config["search"]["candidate_depths"])
print("  Maximum candidate depth:", config["search"]["maximum_candidate_depth"])
print("  Judgement pool depth:", pool_depth)
print("  Expected candidate pairs:", expected_candidate_count)
print("  Expected migrated judgements:", expected_reused_count)
print("  Expected new judgements:", expected_new_count)
print(
    "  Query-embedding API request required:",
    lineage["query_embedding_reuse"]["api_request_required"],
)
print("  Credential value displayed: False")
print("  Question or chunk text displayed: False")
print("  Ground truth accessed: False")
print("Retrieval-evaluation v3 preflight: PASSED")


Retrieval-evaluation v3 preflight:
  Configuration ID: benchmark-retrieval-depth-evaluation-v3
  Questions: 20
  Corpus chunks: 5113
  FAISS vectors: 5113
  Candidate depths: [1, 2, 3, 4, 5, 8, 10, 15, 20, 25, 30]
  Maximum candidate depth: 30
  Judgement pool depth: 40
  Expected candidate pairs: 800
  Expected migrated judgements: 600
  Expected new judgements: 200
  Query-embedding API request required: False
  Credential value displayed: False
  Question or chunk text displayed: False
  Ground truth accessed: False
Retrieval-evaluation v3 preflight: PASSED


## 2. Guarded local v3 preparation

Version 3 reuses the exact version 2 query matrix and searches the existing
FAISS index locally to rank 40. It validates all 600 rank 1-30 candidate pairs
before transferring their existing human grades and notes. The 200 rank 31-40
records remain ungraded.

The production preparation has already been completed and independently
audited. Leave `RUN_V3_PREPARATION = False` during normal notebook execution.
The preparation implementation also refuses to overwrite any existing version
3 artifact.


In [3]:
RUN_V3_PREPARATION = False

if not RUN_V3_PREPARATION:
    print("Retrieval-depth v3 preparation: NOT RUN")
    print("The audited v3 preparation artifacts are reused as they exist.")
else:
    # This branch is available only for a clean first preparation. The
    # implementation refuses to overwrite any existing v3 artifact.
    preparation_result = prepare_retrieval_evaluation_v3(
        CONFIG_PATH,
        LINEAGE_PATH,
        PROJECT_ROOT,
    )
    print("Retrieval-depth v3 preparation: PASSED")
    print("  Candidate pairs:", preparation_result.candidate_count)
    print(
        "  Reused judgements:",
        preparation_result.reused_judgement_count,
    )
    print(
        "  New judgements:",
        preparation_result.new_judgement_count,
    )
    print(
        "  Query matrix SHA-256:",
        preparation_result.query_embedding_matrix_sha256,
    )
    print(
        "  Candidate pool SHA-256:",
        preparation_result.ranked_candidate_pool_sha256,
    )
    print(
        "  Judgement file SHA-256:",
        preparation_result.blinded_judgement_file_sha256,
    )
    print(
        "  Summary SHA-256:",
        preparation_result.summary_sha256,
    )
    print("  API request made: False")
    print("  Ground truth accessed: False")


Retrieval-depth v3 preparation: NOT RUN
The audited v3 preparation artifacts are reused as they exist.


## 3. V3 preparation artifact audit

This cell checks the version 3 shapes, counts, fingerprints and blinded
judgement schema without printing any private content.

The separate tracked preparation audit records that the query matrix is
byte-identical to version 2, all 600 candidate-prefix comparisons passed, all
600 migrated grades and notes remained unchanged, and exactly 200 records were
left for new annotation.


In [4]:
preparation_paths = [
    QUERY_MATRIX_PATH,
    CANDIDATE_POOL_PATH,
    JUDGEMENT_PATH,
    SUMMARY_PATH,
]

if not all(path.is_file() for path in preparation_paths):
    print("Preparation artifact audit: NOT RUN")
    print("Run the guarded paid preparation first.")
else:
    summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    query_matrix = np.load(QUERY_MATRIX_PATH, allow_pickle=False)
    with CANDIDATE_POOL_PATH.open("r", encoding="utf-8") as handle:
        candidate_records = [json.loads(line) for line in handle if line.strip()]
    with JUDGEMENT_PATH.open("r", encoding="utf-8") as handle:
        judgement_records = [json.loads(line) for line in handle if line.strip()]

    expected_judgement_fields = {
        "chunk_text",
        "configuration_id",
        "judgement_id",
        "judgement_note",
        "judgement_schema_version",
        "query_text",
        "relevance_grade",
    }
    hidden_judgement_fields = {
        "question_id",
        "chunk_id",
        "question_position",
        "index_position",
        "rank",
        "similarity_score",
        "parent_record_id",
        "source_id",
        "pdf_page_index",
        "pdf_page_number",
        "printed_page_number",
    }

    if query_matrix.shape != (
        question_count,
        config["query_embedding"]["dimensions"],
    ):
        raise RuntimeError(f"Unexpected query matrix shape: {query_matrix.shape}")
    if query_matrix.dtype != np.float32 or not query_matrix.flags.c_contiguous:
        raise RuntimeError("Query matrix does not follow the float32 C-contiguous contract.")
    if not np.isfinite(query_matrix).all():
        raise RuntimeError("Query matrix contains a non-finite value.")
    if len(candidate_records) != expected_candidate_count:
        raise RuntimeError("Candidate count differs from the frozen protocol.")
    if len(judgement_records) != expected_candidate_count:
        raise RuntimeError("Judgement count differs from the frozen protocol.")
    if any(set(record) != expected_judgement_fields for record in judgement_records):
        raise RuntimeError("A blinded judgement record has unexpected fields.")
    if any(hidden_judgement_fields.intersection(record) for record in judgement_records):
        raise RuntimeError("A rank or provenance field leaked into the judgement file.")
    if summary["question_embedding_matrix_sha256"] != calculate_file_sha256(
        QUERY_MATRIX_PATH
    ):
        raise RuntimeError("Query matrix fingerprint differs from the summary.")
    if summary["ranked_candidate_pool_sha256"] != calculate_file_sha256(
        CANDIDATE_POOL_PATH
    ):
        raise RuntimeError("Candidate-pool fingerprint differs from the summary.")

    if summary.get("query_embedding_reused") is not True:
        raise RuntimeError("The summary does not confirm matrix reuse.")
    if summary.get("prefix_match_passed") is not True:
        raise RuntimeError("The summary does not confirm prefix equality.")
    if (
        summary.get("completed_reused_judgement_count")
        != expected_reused_count
    ):
        raise RuntimeError("The summary has the wrong migrated count.")
    if summary.get("new_unjudged_count") != expected_new_count:
        raise RuntimeError("The summary has the wrong new-judgement count.")
    if summary.get("api_request_made") is not False:
        raise RuntimeError("The summary reports an API request.")

    query_norms = np.linalg.norm(query_matrix, axis=1)
    completed_count = sum(
        record["relevance_grade"] is not None for record in judgement_records
    )

    print("V3 preparation artifact audit:")
    print("  Query matrix shape:", query_matrix.shape)
    print("  Query matrix dtype:", query_matrix.dtype)
    print("  Query matrix C-contiguous:", query_matrix.flags.c_contiguous)
    print("  Finite values:", bool(np.isfinite(query_matrix).all()))
    print("  Minimum norm:", float(query_norms.min()))
    print("  Maximum norm:", float(query_norms.max()))
    print("  Candidate records:", len(candidate_records))
    print("  Blinded judgement records:", len(judgement_records))
    print("  Completed judgements:", completed_count)
    print("  Migrated at preparation:", expected_reused_count)
    print("  New at preparation:", expected_new_count)
    print("  Hidden rank, score and provenance fields present: False")
    print("  Question or chunk text displayed: False")
    print("  Ground truth accessed: False")
    print("V3 preparation artifact audit: PASSED")


V3 preparation artifact audit:
  Query matrix shape: (20, 1536)
  Query matrix dtype: float32
  Query matrix C-contiguous: True
  Finite values: True
  Minimum norm: 0.9999998211860657
  Maximum norm: 1.000000238418579
  Candidate records: 800
  Blinded judgement records: 800
  Completed judgements: 600
  Migrated at preparation: 600
  New at preparation: 200
  Hidden rank, score and provenance fields present: False
  Question or chunk text displayed: False
  Ground truth accessed: False
V3 preparation artifact audit: PASSED


## 4. Blinded relevance judgement helper

The helper displays only the question and one candidate chunk. It does not
display identifiers, retrieval rank, similarity score, candidate-depth
membership or source provenance.

Grades retain the frozen meaning:

- `0`: not useful evidence;
- `1`: useful supporting evidence;
- `2`: direct answer-bearing evidence.

A short evidence note is required for grades 1 and 2. Grade 0 retains a null
note. The helper selects only records whose grade is `None`, so the 600
migrated records are never presented or modified during the additional
annotation.


In [5]:
EXPECTED_JUDGEMENT_FIELDS = {
    "chunk_text",
    "configuration_id",
    "judgement_id",
    "judgement_note",
    "judgement_schema_version",
    "query_text",
    "relevance_grade",
}


def load_local_judgements() -> list[dict[str, object]]:
    """Load the private blinded file without displaying its text."""

    if not JUDGEMENT_PATH.is_file():
        raise RuntimeError("The blinded judgement file does not exist yet.")
    with JUDGEMENT_PATH.open("r", encoding="utf-8") as handle:
        records = [json.loads(line) for line in handle if line.strip()]
    if len(records) != expected_candidate_count:
        raise RuntimeError("Unexpected number of blinded judgement records.")
    if any(set(record) != EXPECTED_JUDGEMENT_FIELDS for record in records):
        raise RuntimeError("A blinded judgement record has unexpected fields.")
    if len({record["judgement_id"] for record in records}) != len(records):
        raise RuntimeError("Blinded judgement identifiers are not unique.")
    return records


def save_local_judgements(records: list[dict[str, object]]) -> None:
    """Atomically replace the private judgement file after validation."""

    if len(records) != expected_candidate_count:
        raise RuntimeError("Refusing to save an incomplete judgement dataset.")
    if any(set(record) != EXPECTED_JUDGEMENT_FIELDS for record in records):
        raise RuntimeError("Refusing to save a record with unexpected fields.")

    JUDGEMENT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=JUDGEMENT_PATH.parent) as temporary_directory:
        temporary_path = Path(temporary_directory) / JUDGEMENT_PATH.name
        temporary_path.write_bytes(serialise_json_lines(records))
        with temporary_path.open("r", encoding="utf-8") as handle:
            reloaded = [json.loads(line) for line in handle if line.strip()]
        if reloaded != records:
            raise RuntimeError("Judgements changed during the save round trip.")
        os.replace(temporary_path, JUDGEMENT_PATH)


def judgement_progress() -> tuple[int, int]:
    """Return completed and total counts without exposing private content."""

    records = load_local_judgements()
    completed = sum(record["relevance_grade"] is not None for record in records)
    return completed, len(records)


def annotation_session(max_items: int = 10) -> None:
    """Judge up to max_items private pairs and save after every response."""

    if isinstance(max_items, bool) or not isinstance(max_items, int) or max_items < 1:
        raise ValueError("max_items must be a positive integer.")

    processed = 0
    try:
        while processed < max_items:
            records = load_local_judgements()
            next_position = next(
                (
                    position
                    for position, record in enumerate(records)
                    if record["relevance_grade"] is None
                ),
                None,
            )
            if next_position is None:
                break

            record = records[next_position]
            completed = sum(
                value["relevance_grade"] is not None for value in records
            )
            clear_output(wait=True)
            display(
                HTML(
                    "<h3>Blinded retrieval judgement</h3>"
                    f"<p>Progress before save: {completed} / {len(records)}</p>"
                    "<h4>Question</h4>"
                    f"<pre style='white-space:pre-wrap'>{escape(str(record['query_text']))}</pre>"
                    "<h4>Candidate chunk</h4>"
                    f"<pre style='white-space:pre-wrap'>{escape(str(record['chunk_text']))}</pre>"
                    "<p><strong>0</strong> irrelevant, "
                    "<strong>1</strong> supporting, "
                    "<strong>2</strong> direct</p>"
                )
            )

            raw_grade = input("Grade 0, 1, 2, or q to stop: ").strip().lower()
            if raw_grade == "q":
                break
            if raw_grade not in {"0", "1", "2"}:
                print("Invalid grade. This pair was not changed.")
                input("Press Enter to continue: ")
                continue

            grade = int(raw_grade)
            note: str | None = None
            if grade in {1, 2}:
                note = input("Short evidence note: ").strip()
                if not note:
                    print("Grades 1 and 2 require a note. This pair was not changed.")
                    input("Press Enter to continue: ")
                    continue

            record["relevance_grade"] = grade
            record["judgement_note"] = note
            save_local_judgements(records)
            processed += 1
    finally:
        clear_output(wait=False)
        if JUDGEMENT_PATH.is_file():
            completed, total = judgement_progress()
            print("Annotation session closed safely.")
            print("  Judgements saved in this session:", processed)
            print("  Completed judgements:", completed)
            print("  Remaining judgements:", total - completed)
            print("  Total judgements:", total)
            print("  Private question or chunk text retained in cell output: False")


print("Blinded annotation helper: READY")
print("No judgement was changed by defining these functions.")


Blinded annotation helper: READY
No judgement was changed by defining these functions.


## 5. Guarded additional annotation session

Change `RUN_ANNOTATION_SESSION` to `True` only while judging a small block.
Ten records per block keeps each save point limited and makes interruptions
easy to recover from.

After each block, restore the flag to `False` and rerun the cell to verify the
saved progress. The initial progress is 600 of 800 because those 600 records
were migrated from the completed version 2 evaluation.


In [69]:
RUN_ANNOTATION_SESSION = False
ANNOTATION_BLOCK_SIZE = 10

if not RUN_ANNOTATION_SESSION:
    print("Blinded annotation session: NOT RUN")
    if JUDGEMENT_PATH.is_file():
        completed, total = judgement_progress()
        print("  Completed judgements:", completed)
        print("  Remaining judgements:", total - completed)
        print("  Total judgements:", total)
else:
    annotation_session(max_items=ANNOTATION_BLOCK_SIZE)


Blinded annotation session: NOT RUN
  Completed judgements: 800
  Remaining judgements: 0
  Total judgements: 800


## 6. V3 completion audit

This audit reports only completion counts and a file fingerprint. It does not
show the grade distribution, question text, chunk text or evidence notes.

Version 3 is complete only when all 800 records have a valid human grade and
every positive grade has its required evidence note.


In [70]:
if not JUDGEMENT_PATH.is_file():
    print("Completion audit: NOT RUN")
    print("The blinded judgement file does not exist yet.")
else:
    completed, total = judgement_progress()
    print("Blinded judgement completion audit:")
    print("  Completed judgements:", completed)
    print("  Remaining judgements:", total - completed)
    print("  Total judgements:", total)
    print("  Current judgement SHA-256:", calculate_file_sha256(JUDGEMENT_PATH))
    print("  Question or chunk text displayed: False")
    print("  Ground truth accessed: False")
    if completed == total:
        print("Blinded judgement completion audit: PASSED")
    else:
        print("Blinded judgement completion audit: INCOMPLETE")


Blinded judgement completion audit:
  Completed judgements: 800
  Remaining judgements: 0
  Total judgements: 800
  Current judgement SHA-256: cbe0868da7c7637110de0f7aa57f3c4276939fde245fa1c2b8cf2556867366fe
  Question or chunk text displayed: False
  Ground truth accessed: False
Blinded judgement completion audit: PASSED


## 7. Guarded offline v3 scoring

Scoring is allowed only after all 800 judgements are complete. It validates
the private candidate and judgement files, calculates candidate depths 1, 2,
3, 4, 5, 8, 10, 15, 20, 25 and 30, and applies the unchanged selection rule:

1. maximise the number of questions with direct evidence;
2. if tied, maximise the number with useful evidence;
3. if still tied, select the smallest depth.

The version 3 adequacy gate triggers if a question without direct evidence in
ranks 1-30 gains direct evidence in ranks 31-40. If it triggers, no retrieval
depth is selected. Version 3 is the final planned expansion, so a triggered
gate leads to investigation of the retrieval method instead of an automatic
version 4 range extension.

This step makes no API request and does not load ground-truth answers. Leave
the scoring flag disabled until the completion audit passes.


In [ ]:
RUN_OFFLINE_SCORING = False

if not RUN_OFFLINE_SCORING:
    print("Offline retrieval scoring: NOT RUN")
    print("Set RUN_OFFLINE_SCORING = True only after all judgements are complete.")
else:
    completed, total = judgement_progress()
    if completed != total:
        raise RuntimeError(
            f"Cannot score incomplete judgements: {completed} / {total} complete."
        )
    metrics_result = score_retrieval_judgements(
        CONFIG_PATH,
        PROJECT_ROOT,
        overwrite=False,
    )
    print("Offline retrieval scoring: PASSED")
    print("  Completed judgements:", metrics_result.completed_judgement_count)
    print(
        "  Candidate-range adequacy gate triggered:",
        metrics_result.candidate_range_adequacy_gate_triggered,
    )
    print("  Selected retrieval depth k:", metrics_result.selected_depth_k)
    print("  Metrics:", metrics_result.metrics_path.relative_to(PROJECT_ROOT))
    print("  Metrics SHA-256:", metrics_result.metrics_sha256)
    print("  Metrics table:", metrics_result.metrics_table_path.relative_to(PROJECT_ROOT))
    print("  Metrics table SHA-256:", metrics_result.metrics_table_sha256)
    print("  API request made during scoring: False")
    print("  Ground truth accessed: False")


Offline retrieval scoring: PASSED
  Completed judgements: 800
  Candidate-range adequacy gate triggered: False
  Selected retrieval depth k: 30
  Metrics: results/metrics/retrieval-depth-evaluation-v3.json
  Metrics SHA-256: 4415459e535e832a84d87a468eb4ced21660be8489bfbb22248cf84f3ed3c2b4
  Metrics table: results/tables/retrieval-depth-metrics-v3.csv
  Metrics table SHA-256: 306baa7eb77ae06ad019a1f9976f1110944260d4cbae67dc6206c41cf7bb78cd
  API request made during scoring: False
  Ground truth accessed: False


## 8. Text-free v3 metric review

After scoring, this cell displays the aggregated version 3 metric rows,
adequacy-gate status and selection result. The tracked results contain only
aggregated values, identifiers needed for text-free diagnostics and artifact
fingerprints.

No question text, chunk text, evidence note or ground-truth answer is shown.


In [72]:
if not METRICS_PATH.is_file() or not METRICS_TABLE_PATH.is_file():
    print("Metric review: NOT RUN")
    print("Run guarded offline scoring after completing every judgement.")
else:
    metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
    metrics_text = METRICS_PATH.read_text(encoding="utf-8")

    if "query_text" in metrics_text or "chunk_text" in metrics_text:
        raise RuntimeError("Tracked metrics contain a forbidden text field name.")

    print("Retrieval-depth metric review:")
    print("  Selected retrieval depth k:", metrics["selected_depth_k"])
    print("  Selection status:", metrics["selection_status"])
    print(
        "  Candidate-range adequacy gate triggered:",
        metrics["candidate_range_adequacy_gate"]["triggered"],
    )
    print("  Completed judgements:", metrics["completed_judgement_count"])
    print("  Metrics SHA-256:", calculate_file_sha256(METRICS_PATH))
    print("  Metrics table SHA-256:", calculate_file_sha256(METRICS_TABLE_PATH))
    print("  Question or chunk text displayed: False")
    print("  Ground truth accessed: False")
    print()
    print(METRICS_TABLE_PATH.read_text(encoding="utf-8"))
    print("Retrieval-depth metric review: PASSED")


Retrieval-depth metric review:
  Selected retrieval depth k: 30
  Selection status: selected_by_frozen_rule
  Candidate-range adequacy gate triggered: False
  Completed judgements: 800
  Metrics SHA-256: 4415459e535e832a84d87a468eb4ced21660be8489bfbb22248cf84f3ed3c2b4
  Metrics table SHA-256: 306baa7eb77ae06ad019a1f9976f1110944260d4cbae67dc6206c41cf7bb78cd
  Question or chunk text displayed: False
  Ground truth accessed: False

depth_k,direct_evidence_hit_count,direct_evidence_hit_rate,direct_evidence_hit_rate_wilson_95_lower,direct_evidence_hit_rate_wilson_95_upper,useful_evidence_hit_count,useful_evidence_hit_rate,useful_evidence_hit_rate_wilson_95_lower,useful_evidence_hit_rate_wilson_95_upper,direct_evidence_mrr,graded_ndcg,useful_evidence_precision,mean_retrieved_context_tokens,mean_retrieved_context_characters
1,3,0.15,0.05236779195949584,0.36042329588695743,17,0.85,0.6395767041130426,0.9476322080405041,0.15,0.4166666666666667,0.85,69.8,183.55
2,5,0.25,0.11186005278940309,0.4687

## 9. Reproducibility and reporting boundary

Version 3 is an adaptive extension created after the version 2 adequacy gate
was observed. It must not be described as part of the original version 1
protocol.

If the version 3 adequacy gate does not trigger, the unchanged frozen
selection rule chooses the smallest depth that preserves the maximum direct
and useful evidence coverage. If the gate triggers, the result remains
unresolved and the next methodological step is to investigate retrieval
quality rather than automatically expanding the depth again.

The dissertation should report:

- why version 3 was introduced;
- that the exact query matrix and first 600 human judgements were reused;
- the candidate depths through 30;
- the rank 31-40 diagnostic tail;
- the exact migration and blinding controls;
- the terminal adequacy-gate result;
- the context-size trade-off across candidate depths;
- the limitation from using the same 20 questions for calibration and later
  model evaluation.

Before committing any executed notebook state:

- restore `RUN_V3_PREPARATION`, `RUN_ANNOTATION_SESSION` and
  `RUN_OFFLINE_SCORING` to `False`;
- remove any private annotation prompt from saved output;
- retain only safe, text-free audit output;
- confirm that no error output remains;
- confirm that no ground-truth answer was accessed.
